# Experiment 2: Counterfactual Sensitivity

Family-level analysis: fragility rate, severity distribution, pattern analysis.

**Primary analysis unit**: family (not instance). Family members are correlated, not IID.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from bcbench.analysis.family import FamilyOutcome, FamilyType
from bcbench.analysis.family_aggregator import build_families
from evaluator.thesis_metrics import (
    family_type_distribution,
    fragility_rate,
    mean_severity,
)
from utils import load_all_results

# Configure after running experiments
SETUPS: list[str] = []  # e.g. ["copilot-cf", "claude-cf"]
SETUP_LABELS: dict[str, str] = {}

# Load CF results
all_results = load_all_results(category="bug-fix")  # CF results stored under bug-fix
print(f"Loaded {len(all_results)} setups")

## Family Outcome Table

Each row = one family. Columns: family_id, layer, base, cf1..cfN, pattern, type, fragile, severity.

In [ ]:
# Placeholder: build families from results
# When results are available, replace with actual result loading:
#   results = load_evaluation_results(setup_path)
#   families = build_families(results, failure_layers=layer_map)

# Example with mock data for development:
from bcbench.analysis.family import InstanceResult
from bcbench.types import FailureLayer


def families_to_df(families: list[FamilyOutcome]) -> pd.DataFrame:
    rows = []
    for f in families:
        row = {
            "family_id": f.family_id,
            "layer": f.failure_layer.value if f.failure_layer else "unclassified",
            "base": int(f.base.passed),
            "pattern": str(f.pattern),
            "type": f.family_type.value,
            "fragile": int(f.is_fragile),
            "severity": f.severity,
            "cf_total": f.cf_total,
            "cf_fail_count": f.cf_fail_count,
        }
        for i, cf in enumerate(f.cfs):
            row[f"cf{i+1}"] = int(cf.passed)
        rows.append(row)
    return pd.DataFrame(rows)


print("Family outcome table builder ready. Load results to populate.")

## Family Type Distribution

In [ ]:
def plot_family_type_distribution(families_by_model: dict[str, list[FamilyOutcome]]):
    rows = []
    for model, families in families_by_model.items():
        dist = family_type_distribution(families)
        total = sum(dist.values())
        for ftype, count in dist.items():
            rows.append({
                "Model": model,
                "Family Type": ftype,
                "Count": count,
                "Proportion (%)": round(count / total * 100, 1) if total else 0,
            })

    df = pd.DataFrame(rows)
    fig = px.bar(
        df, x="Model", y="Proportion (%)", color="Family Type",
        title="Family Type Distribution by Model",
        barmode="stack", text_auto=True,
        color_discrete_map={
            "stable-correct": "#2ecc71",
            "fragile": "#e74c3c",
            "unsolved": "#95a5a6",
            "inconsistent": "#f39c12",
        },
    )
    fig.show()
    return df


print("plot_family_type_distribution() ready")

## Fragility Rate (Main Thesis Figure)

In [ ]:
def plot_fragility_rate(families_by_model: dict[str, list[FamilyOutcome]]):
    rows = []
    for model, families in families_by_model.items():
        rate = fragility_rate(families)
        sev = mean_severity(families)
        rows.append({
            "Model": model,
            "Fragility Rate (%)": round(rate * 100, 1),
            "Mean Severity": round(sev, 3) if sev is not None else None,
        })

    df = pd.DataFrame(rows)
    fig = px.bar(
        df, x="Model", y="Fragility Rate (%)",
        title="Family Fragility Rate: P(CF fail | Base correct)",
        text_auto=True,
        color_discrete_sequence=["#e74c3c"],
    )
    fig.update_layout(yaxis_range=[0, 100])
    fig.show()
    return df


print("plot_fragility_rate() ready")

## Severity Distribution

In [ ]:
def plot_severity_distribution(families_by_model: dict[str, list[FamilyOutcome]]):
    rows = []
    for model, families in families_by_model.items():
        for f in families:
            if f.severity is not None:
                rows.append({"Model": model, "Severity": f.severity, "Type": f.family_type.value})

    df = pd.DataFrame(rows)
    if not df.empty:
        fig = px.box(
            df, x="Model", y="Severity", color="Model",
            title="Fragility Severity Distribution (N_cf_fail / N_cf)",
            points="all",
        )
        fig.show()
    return df


print("plot_severity_distribution() ready")

## Pattern Analysis

In [ ]:
def analyze_patterns(families: list[FamilyOutcome]) -> pd.DataFrame:
    """Count and rank the most common family outcome patterns."""
    from collections import Counter
    pattern_counts = Counter(f.pattern for f in families)
    rows = [
        {"Pattern": str(p), "Count": c, "Pct (%)": round(c / len(families) * 100, 1)}
        for p, c in pattern_counts.most_common()
    ]
    return pd.DataFrame(rows)


print("analyze_patterns() ready")